In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef
import matplotlib.pyplot as plt
import time
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, TensorDataset

# Model class (unchanged)
class ComplexMultiHeadSelfAttentionClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=4, num_heads=8, hidden_dim=512, num_layers=3,
                 use_attention=True, use_ffn=True, use_dropout=True):
        super().__init__()
        self.use_attention = use_attention
        self.use_ffn = use_ffn
        self.use_dropout = use_dropout
        
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2) if use_dropout else nn.Identity()
        )
        
        if use_attention:
            self.attention_layers = nn.ModuleList([
                nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, 
                                    dropout=0.1 if use_dropout else 0.0)
                for _ in range(num_layers)
            ])
        else:
            self.attention_layers = None
            
        if use_ffn:
            self.ffn = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.ReLU(),
                nn.Dropout(0.2) if use_dropout else nn.Identity(),
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.LayerNorm(hidden_dim)
            )
        else:
            self.ffn = nn.Identity()
            
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2) if use_dropout else nn.Identity(),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        
    def forward(self, x):
        x = x.unsqueeze(0)
        x = self.embedding(x)
        if self.use_attention and self.attention_layers is not None:
            for attn in self.attention_layers:
                attn_output, _ = attn(x, x, x)
                x = x + attn_output
                x = self.ffn(x) if self.use_ffn else x
        else:
            x = self.ffn(x) if self.use_ffn else x
        x = x.squeeze(0)
        x = self.classifier(x)
        return x

# Load and prepare data
data = pd.read_csv('mrmr_selected_1000_features_with_labels.csv')
X = data.drop(['label', 'class'], axis=1)
y = data['label']
class_names = data['class'].unique().tolist()
n_classes = len(class_names)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train_tensor = torch.FloatTensor(X_train.values).to(device)
X_test_tensor = torch.FloatTensor(X_test.values).to(device)
y_train_tensor = torch.LongTensor(y_train.values).to(device)
y_test_tensor = torch.LongTensor(y_test.values).to(device)

# Function to calculate specificity from confusion matrix
def calculate_specificity(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred)
    specificity = []
    for i in range(n_classes):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        specificity.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    return np.mean(specificity)

# Training and evaluation function
def train_and_evaluate(config_name, use_attention=True, use_ffn=True, use_dropout=True, hidden_dim=512):
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    
    model = ComplexMultiHeadSelfAttentionClassifier(
        input_dim=1000,
        num_classes=n_classes,
        num_heads=8,
        hidden_dim=hidden_dim,
        num_layers=3,
        use_attention=use_attention,
        use_ffn=use_ffn,
        use_dropout=use_dropout
    ).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)
    scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
    criterion = nn.CrossEntropyLoss()
    
    # Training
    start_time = time.time()
    model.train()
    for epoch in range(100):
        total_loss = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        scheduler.step(avg_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/100, Config: {config_name}, Avg Loss: {avg_loss:.4f}")
    
    training_time = time.time() - start_time
    
    # Evaluation
    start_time = time.time()
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        y_pred = torch.argmax(outputs, dim=1)
        y_pred_proba = torch.softmax(outputs, dim=1)
    testing_time = time.time() - start_time
    
    y_test_np = y_test_tensor.cpu().numpy()
    y_pred_np = y_pred.cpu().numpy()
    y_pred_proba_np = y_pred_proba.cpu().numpy()
    
    # Calculate all metrics
    metrics = {
        'config': config_name,
        'ACC': accuracy_score(y_test_np, y_pred_np),
        'AUC': roc_auc_score(y_test_np, y_pred_proba_np, multi_class='ovr'),
        'PRE': precision_score(y_test_np, y_pred_np, average='macro'),
        'SP': calculate_specificity(y_test_np, y_pred_np, n_classes),
        'SN': recall_score(y_test_np, y_pred_np, average='macro'),
        'F1': f1_score(y_test_np, y_pred_np, average='macro'),
        'MCC': matthews_corrcoef(y_test_np, y_pred_np),
        'training_time': training_time,
        'testing_time': testing_time,
        'confusion_matrix': confusion_matrix(y_test_np, y_pred_np).tolist()
    }
    
    return metrics, y_pred_proba_np

# Ablation study configurations
configs = [
    ("Full_Model", True, True, True, 512),
    ("No_Attention", False, True, True, 512),
    ("No_FFN", True, False, True, 512),
    ("No_Dropout", True, True, False, 512),
    ("Reduced_Hidden", True, True, True, 256)
]

# Run experiments
results = []
y_pred_probas = []
method_names = []

for config_name, use_attn, use_ffn, use_drop, hidden_dim in configs:
    print(f"\nTraining {config_name} (AdamW, lr=0.0001, bs=64, 100 epochs)...")
    metrics, y_pred_proba = train_and_evaluate(config_name, use_attn, use_ffn, use_drop, hidden_dim)
    results.append(metrics)
    y_pred_probas.append(y_pred_proba)
    method_names.append(config_name)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv('ablation_study_results_adamw_lr0001_bs64_100epochs.csv', index=False)

# ROC Curve plotting functions
def plot_roc_curves(y_test, y_pred_probas, class_names, method_names, n_classes):
    plt.figure(figsize=(12, 10), facecolor='white')
    plt.rcParams['font.family'] = 'Times New Roman'
    colors = ['blue', 'green', 'red', 'purple', 'orange']
    for m_idx, (method_name, y_pred_proba) in enumerate(zip(method_names, y_pred_probas)):
        for i in range(n_classes):
            y_true_binary = (y_test == i).astype(int)
            fpr, tpr, _ = roc_curve(y_true_binary, y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, color=colors[m_idx], lw=3,
                    label=f'{method_name} - {class_names[i]} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=24)
    plt.ylabel('True Positive Rate', fontsize=24)
    plt.title('ROC Curves - Ablation Study (AdamW, lr=0.0001, bs=64)', fontsize=28, pad=20)
    plt.legend(loc="lower right", fontsize=16)
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    for spine in plt.gca().spines.values():
        spine.set_linewidth(2)
        spine.set_edgecolor('black')
    plt.savefig('roc_curves_ablation_adamw_lr0001_bs64_100epochs.png', dpi=1000, bbox_inches='tight')
    plt.savefig('roc_curves_ablation_adamw_lr0001_bs64_100epochs.pdf', dpi=1000, bbox_inches='tight')
    plt.close()

def plot_roc_curves_by_class(y_test, y_pred_probas, class_names, method_names, n_classes):
    colors = ['blue', 'green', 'red', 'purple', 'orange']
    for i in range(n_classes):
        plt.figure(figsize=(8, 8), facecolor='white')
        plt.rcParams['font.family'] = 'Times New Roman'
        for m_idx, (method_name, y_pred_proba) in enumerate(zip(method_names, y_pred_probas)):
            y_true_binary = (y_test == i).astype(int)
            fpr, tpr, _ = roc_curve(y_true_binary, y_pred_proba[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, color=colors[m_idx], lw=3,
                    label=f'{method_name} (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--', lw=2)
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate', fontsize=24)
        plt.ylabel('True Positive Rate', fontsize=24)
        plt.title(f'ROC Curve - {class_names[i]} (AdamW, lr=0.0001, bs=64)', fontsize=28, pad=20)
        plt.legend(loc="lower right", fontsize=16)
        plt.xticks(fontsize=18)
        plt.yticks(fontsize=18)
        for spine in plt.gca().spines.values():
            spine.set_linewidth(2)
            spine.set_edgecolor('black')
        plt.savefig(f'roc_curve_class_{i}_ablation_adamw_lr0001_bs64_100epochs.png', dpi=1000, bbox_inches='tight')
        plt.savefig(f'roc_curve_class_{i}_ablation_adamw_lr0001_bs64_100epochs.pdf', dpi=1000, bbox_inches='tight')
        plt.close()

# Generate plots
y_test_np = y_test_tensor.cpu().numpy()
plot_roc_curves(y_test_np, y_pred_probas, class_names, method_names, n_classes)
plot_roc_curves_by_class(y_test_np, y_pred_probas, class_names, method_names, n_classes)

# Print results
print("\nAblation Study Results (AdamW, lr=0.0001, batch size 64, 100 epochs):")
for result in results:
    print(f"\nConfiguration: {result['config']}")
    print(f"Accuracy (ACC): {result['ACC']:.4f}")
    print(f"AUC: {result['AUC']:.4f}")
    print(f"Precision (PRE): {result['PRE']:.4f}")
    print(f"Specificity (SP): {result['SP']:.4f}")
    print(f"Sensitivity (SN): {result['SN']:.4f}")
    print(f"F1-score (F1): {result['F1']:.4f}")
    print(f"Matthews CC (MCC): {result['MCC']:.4f}")
    print(f"Training Time: {result['training_time']:.2f}s")
    print(f"Testing Time: {result['testing_time']:.2f}s")
    print("Confusion Matrix:")
    print(np.array(result['confusion_matrix']))

print("\nResults saved to 'ablation_study_results_adamw_lr0001_bs64_100epochs.csv'")
print("ROC curves saved as PNG and PDF files")